# H&E ↔ PSR WSI Registration Pipeline

Converts flat `.tif` files to pyramidal OME-TIFFs, then registers each H&E / PSR pair.

**Expected folder structure** — H&E and PSR files live in separate directories:
```
HE_DIR/
├── mouse01.tif
├── mouse02.tif
└── mouse03.tif

PSR_DIR/
├── mouse01.tif
├── mouse02.tif
└── mouse03.tif
```
Files are paired by matching their stems. If the stems contain a stain keyword
(e.g. `mouse01_HE.tif` / `mouse01_PSR.tif`), the keyword is stripped before matching.

## 1 — Configuration

In [ ]:
# ── Edit these before running ─────────────────────────────────────────────────

HE_DIR        = "/path/to/HE/files"         # folder containing H&E .tif files
PSR_DIR       = "/path/to/PSR/files"        # folder containing PSR .tif files
OUTPUT_DIR    = "./registration_output"      # registered results go here
PIXEL_SIZE_UM = 0.23                         # µm/px — check your scanner metadata

# Optional: keywords in filenames used to identify each stain (case-insensitive).
# Set to "" if filenames carry no stain keyword and stems match directly.
HE_KEYWORD    = "HE"
PSR_KEYWORD   = "PSR"

# Pyramid conversion
PYRAMID_LEVELS = 6     # number of resolution levels (6 → ~64× downsample)
TILE_SIZE      = 512   # tile size in pixels; must be a multiple of 16

# wsireg preprocessing — green channel + contrast enhancement for both stains
HE_PREPRO  = {"as_uint8": True, "ch_indices": [1], "contrast_enhance": True}
PSR_PREPRO = {"as_uint8": True, "ch_indices": [1], "contrast_enhance": True}

## 2 — Imports

In [ ]:
import re
from pathlib import Path

import numpy as np
import tifffile
import matplotlib.pyplot as plt
from PIL import Image
from wsireg.wsireg2d import WsiReg2D
import openslide

## 3 — Discover and pair files

In [ ]:
def collect_tifs(directory: str) -> list:
    d = Path(directory)
    return sorted(d.glob("*.tif")) + sorted(d.glob("*.tiff"))


def strip_keyword(stem: str, keyword: str) -> str:
    """Remove a stain keyword and surrounding separators from a filename stem."""
    if not keyword:
        return stem
    return re.sub(
        rf"[_\-.]?{re.escape(keyword)}[_\-.]?", "", stem, flags=re.IGNORECASE
    ).strip("_-.")


he_files  = collect_tifs(HE_DIR)
psr_files = collect_tifs(PSR_DIR)

# Build base_name → file lookup for each stain
he_map  = {strip_keyword(f.stem, HE_KEYWORD):  f for f in he_files}
psr_map = {strip_keyword(f.stem, PSR_KEYWORD): f for f in psr_files}

common_keys = sorted(set(he_map) & set(psr_map))
pairs = [(he_map[k], psr_map[k]) for k in common_keys]

print(f"H&E  directory : {HE_DIR}  ({len(he_files)} files)")
print(f"PSR  directory : {PSR_DIR}  ({len(psr_files)} files)")
print(f"Matched pairs  : {len(pairs)}\n")

for he, psr in pairs:
    print(f"  HE : {he.name}")
    print(f"  PSR: {psr.name}")
    print()

unmatched_he  = set(he_map)  - set(psr_map)
unmatched_psr = set(psr_map) - set(he_map)
if unmatched_he:
    print(f"Warning — unmatched H&E : {unmatched_he}")
if unmatched_psr:
    print(f"Warning — unmatched PSR : {unmatched_psr}")

## 4 — Convert flat .tif → pyramidal OME-TIFF

In [ ]:
def build_pyramid(img: np.ndarray, n_levels: int) -> list:
    levels = [img]
    for _ in range(n_levels - 1):
        prev = levels[-1]
        h, w = prev.shape[:2]
        prev = prev[: h - h % 2, : w - w % 2]
        if prev.ndim == 3:
            down = (
                prev[0::2, 0::2].astype(np.uint32) + prev[1::2, 0::2]
                + prev[0::2, 1::2] + prev[1::2, 1::2]
            ) // 4
        else:
            down = (
                prev[0::2, 0::2].astype(np.uint32) + prev[1::2, 0::2]
                + prev[0::2, 1::2] + prev[1::2, 1::2]
            ) // 4
        levels.append(down.astype(img.dtype))
        if min(down.shape[:2]) < 512:
            break
    return levels


def convert_to_pyramid(src: Path, dst: Path, n_levels: int, tile_size: int) -> Path:
    if dst.exists():
        print(f"  [skip] {dst.name} already exists")
        return dst

    print(f"  Reading  {src.name} …")
    img = tifffile.imread(str(src))
    print(f"  Shape: {img.shape}  dtype: {img.dtype}")

    levels = build_pyramid(img, n_levels)
    photometric = "rgb" if img.ndim == 3 and img.shape[2] == 3 else "minisblack"

    print(f"  Writing  {dst.name} ({len(levels)} levels) …")
    with tifffile.TiffWriter(str(dst), bigtiff=True) as tw:
        opts = dict(photometric=photometric, tile=(tile_size, tile_size),
                    compression="deflate", metadata=None)
        tw.write(levels[0], subfiletype=0, subifds=len(levels) - 1, **opts)
        for lvl in levels[1:]:
            tw.write(lvl, subfiletype=1, **opts)
    return dst


# Converted pyramids are stored in two sub-folders to keep stains separate
pyramid_he_dir  = Path(OUTPUT_DIR) / "pyramids" / "HE"
pyramid_psr_dir = Path(OUTPUT_DIR) / "pyramids" / "PSR"
pyramid_he_dir.mkdir(parents=True, exist_ok=True)
pyramid_psr_dir.mkdir(parents=True, exist_ok=True)

converted_pairs = []
for he_src, psr_src in pairs:
    print(f"\nPair: {he_src.stem} / {psr_src.stem}")
    he_dst  = convert_to_pyramid(he_src,  pyramid_he_dir  / (he_src.stem  + ".ome.tiff"), PYRAMID_LEVELS, TILE_SIZE)
    psr_dst = convert_to_pyramid(psr_src, pyramid_psr_dir / (psr_src.stem + ".ome.tiff"), PYRAMID_LEVELS, TILE_SIZE)
    converted_pairs.append((he_dst, psr_dst))

print("\nConversion complete.")

## 5 — Register each pair

In [ ]:
def register_pair(he_path: Path, psr_path: Path, out_dir: Path, pixel_size: float) -> Path:
    project_name = strip_keyword(he_path.stem.replace(".ome", ""), HE_KEYWORD)
    pair_out     = out_dir / project_name
    pair_out.mkdir(parents=True, exist_ok=True)

    existing = list(pair_out.glob("*.ome.tiff"))
    if existing:
        print(f"  [skip] {project_name} already registered → {existing[0].name}")
        return existing[0]

    reg = WsiReg2D(project_name, str(pair_out))
    reg.add_modality("HE",  str(he_path),  image_res=pixel_size, prepro_dict=HE_PREPRO)
    reg.add_modality("PSR", str(psr_path), image_res=pixel_size, prepro_dict=PSR_PREPRO)
    reg.add_reg_path("PSR", "HE", reg_params=["rigid", "affine", "nl"])

    reg.register_images()
    reg.transform_images(file_writer="ome.tiff")

    return sorted(pair_out.glob("*.ome.tiff"))[0]


reg_out_dir        = Path(OUTPUT_DIR) / "registered"
registered_results = []   # list of (he, psr, registered_psr)

for i, (he, psr) in enumerate(converted_pairs):
    print(f"\n[{i+1}/{len(converted_pairs)}] Registering {psr.stem} → H&E space …")
    reg_psr = register_pair(he, psr, reg_out_dir, PIXEL_SIZE_UM)
    registered_results.append((he, psr, reg_psr))
    print(f"  Done → {reg_psr}")

print("\nAll registrations complete.")

## 6 — QC checkerboard overlays

In [ ]:
QC_LEVEL = 5   # pyramid level for thumbnails; increase number if too slow

def read_wsi_thumb(path: Path, level: int) -> np.ndarray:
    slide  = openslide.OpenSlide(str(path))
    actual = min(level, slide.level_count - 1)
    w, h   = slide.level_dimensions[actual]
    return np.array(slide.read_region((0, 0), actual, (w, h)).convert("RGB"))


def read_ometiff_thumb(path: Path, level: int) -> np.ndarray:
    with tifffile.TiffFile(str(path)) as tif:
        series = tif.series[0]
        actual = min(level, len(series.levels) - 1)
        arr    = series.levels[actual].asarray()
    if arr.ndim == 3 and arr.shape[0] in (1, 3, 4):
        arr = np.moveaxis(arr, 0, -1)
    return arr[..., :3].astype(np.uint8)


def checkerboard(a: np.ndarray, b: np.ndarray, n: int = 8) -> np.ndarray:
    h, w   = a.shape[:2]
    th, tw = h // n, w // n
    board  = a.copy()
    for r in range(n):
        for c in range(n):
            if (r + c) % 2 == 0:
                board[r*th:(r+1)*th, c*tw:(c+1)*tw] = b[r*th:(r+1)*th, c*tw:(c+1)*tw]
    return board


qc_dir = Path(OUTPUT_DIR) / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)

for he, psr, reg_he in registered_results:
    sample_name = strip_keyword(he.stem.replace(".ome", ""), HE_KEYWORD)

    psr_arr     = read_wsi_thumb(psr, QC_LEVEL)
    orig_he_arr = read_wsi_thumb(he,  QC_LEVEL)
    try:
        reg_he_arr = read_ometiff_thumb(reg_he, QC_LEVEL)
    except Exception:
        reg_he_arr = orig_he_arr

    h, w        = psr_arr.shape[:2]
    orig_he_arr = np.array(Image.fromarray(orig_he_arr).resize((w, h), Image.LANCZOS))
    reg_he_arr  = np.array(Image.fromarray(reg_he_arr).resize((w, h),  Image.LANCZOS))
    board       = checkerboard(psr_arr, reg_he_arr)

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle(sample_name, fontsize=13, fontweight="bold")
    for ax, img, title in zip(
        axes,
        [psr_arr, orig_he_arr, reg_he_arr, board],
        ["PSR (fixed)", "H&E (original)", "H&E (registered)", "Checkerboard"],
    ):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    out_png = qc_dir / f"{sample_name}_qc.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_png}")

## 7 — Summary

In [ ]:
print("═" * 60)
print(f"  Pairs processed : {len(registered_results)}")
print(f"  Pyramids (H&E)  : {Path(OUTPUT_DIR) / 'pyramids' / 'HE'}")
print(f"  Pyramids (PSR)  : {Path(OUTPUT_DIR) / 'pyramids' / 'PSR'}")
print(f"  Registered WSIs : {Path(OUTPUT_DIR) / 'registered'}")
print(f"  QC overlays     : {Path(OUTPUT_DIR) / 'qc'}")
print("═" * 60)
print()
for he, psr, reg_he in registered_results:
    sample = strip_keyword(he.stem.replace(".ome", ""), HE_KEYWORD)
    print(f"  {sample}")
    print(f"    H&E (registered) → {reg_he.name}")